# Retail Demand Forecasting — Exploratory Data Analysis

> **Project:** End-to-end demand forecasting for a leading Bolivian home improvement retailer  
> **Author:** David Bravo · [bravoaidatastudio.com](https://bravoaidatastudio.com)  
> **Data period:** January 2023 – December 2025 (3 years, daily granularity)  
> **Scope:** 4 stores across La Paz, Santa Cruz, Cochabamba and El Alto | 30 SKUs across 8 product categories

---

## Notebook Structure

1. Data Loading & Quality Audit  
2. Univariate Analysis — Sales Distribution  
3. Temporal Patterns — Day, Week, Month, Year  
4. Bolivian Seasonality — Rainy vs. Dry Season  
5. Holiday Effect Analysis  
6. Category-Level Deep Dive  
7. Store Performance Comparison  
8. Correlation & Feature Relationships  
9. Key Findings & Modeling Implications


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})
PALETTE = ['#2E86AB', '#E84855', '#F4A261', '#2A9D8F', '#8338EC', '#FB5607', '#3A86FF', '#06D6A0']
sns.set_palette(PALETTE)

print('Libraries loaded successfully.')

---
## 1. Data Loading & Quality Audit

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
DATA_PATH = Path('../data/raw_sales.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Dataset shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range    : {df.date.min().date()} → {df.date.max().date()}')
print(f'Unique SKUs   : {df.sku_id.nunique()}')
print(f'Unique stores : {df.store_id.nunique()}')
print(f'Categories    : {df.category.nunique()}')
df.head()

In [ ]:
# ── Data quality audit ────────────────────────────────────────────
print('=== DATA QUALITY REPORT ===')
print(f"\nMissing values:")
print(df.isnull().sum())

print(f"\nNegative quantities: {(df.quantity_sold < 0).sum()}")
print(f"Zero-sale records  : {(df.quantity_sold == 0).sum():,}  ({(df.quantity_sold == 0).mean()*100:.1f}% of dataset)")

print(f"\nQuantity sold — descriptive stats:")
df['quantity_sold'].describe().round(2)

---
## 2. Univariate Analysis — Sales Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Sales Distribution — Daily Quantity Sold per SKU×Store', fontsize=14, fontweight='bold', y=1.02)

# Histogram
non_zero = df[df['quantity_sold'] > 0]['quantity_sold']
axes[0].hist(non_zero, bins=60, color=PALETTE[0], edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0].axvline(non_zero.median(), color=PALETTE[1], lw=2, linestyle='--', label=f'Median: {non_zero.median():.0f}')
axes[0].axvline(non_zero.mean(), color=PALETTE[2], lw=2, linestyle='--', label=f'Mean: {non_zero.mean():.1f}')
axes[0].set_xlabel('Units sold per day (non-zero records)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Histogram')
axes[0].legend()

# Box plot per category
cat_order = df.groupby('category')['quantity_sold'].median().sort_values(ascending=False).index
df_plot = df[df['quantity_sold'] > 0]
axes[1].boxplot(
    [df_plot[df_plot['category'] == cat]['quantity_sold'].values for cat in cat_order],
    labels=cat_order, patch_artist=True,
    boxprops=dict(facecolor=PALETTE[0], alpha=0.7),
    medianprops=dict(color=PALETTE[1], linewidth=2),
    flierprops=dict(marker='.', markersize=3, alpha=0.4)
)
axes[1].set_xticklabels(cat_order, rotation=35, ha='right')
axes[1].set_ylabel('Units sold')
axes[1].set_title('Spread by Category')

plt.tight_layout()
plt.savefig('../data/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Temporal Patterns

In [ ]:
# ── Aggregate: total daily sales across all SKUs and stores ───────
daily = df.groupby('date')['quantity_sold'].sum().reset_index()
daily['rolling_28d'] = daily['quantity_sold'].rolling(28, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(daily['date'], daily['quantity_sold'], alpha=0.18, color=PALETTE[0])
ax.plot(daily['date'], daily['quantity_sold'], lw=0.6, color=PALETTE[0], alpha=0.6, label='Daily total')
ax.plot(daily['date'], daily['rolling_28d'], lw=2, color=PALETTE[1], label='28-day rolling mean')

# Annotate key events
events = {
    '2023-08-06': ('Independencia 2023', 0.02),
    '2024-06-21': ('Año Nuevo Aymara', 0.02),
    '2024-08-06': ('Independencia 2024', 0.02),
    '2025-08-06': ('Independencia 2025', 0.02),
}
for date_str, (label, offset) in events.items():
    xdate = pd.to_datetime(date_str)
    yval = daily.loc[daily['date'] == xdate, 'quantity_sold'].values
    if len(yval):
        ax.annotate(label, xy=(xdate, yval[0]), xytext=(xdate, yval[0] * 1.12),
                    fontsize=8, color='#444',
                    arrowprops=dict(arrowstyle='->', color='#888', lw=1.2))

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
ax.set_title('Total Daily Sales — All SKUs & Stores (Jan 2023 – Dec 2025)', fontsize=13, fontweight='bold')
ax.set_ylabel('Total units sold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/eda_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Day-of-week and monthly patterns ─────────────────────────────
df['day_of_week'] = df['date'].dt.day_name()
df['month_name']  = df['date'].dt.strftime('%b')
df['month_num']   = df['date'].dt.month
df['year']        = df['date'].dt.year

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Temporal Sales Patterns', fontsize=14, fontweight='bold')

# Day of week
dow_avg = df.groupby('day_of_week')['quantity_sold'].mean().reindex(dow_order)
bars = axes[0].bar(dow_order, dow_avg.values, color=PALETTE[:7], edgecolor='white', linewidth=0.5)
axes[0].bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
axes[0].set_title('Average Daily Sales by Day of Week')
axes[0].set_ylabel('Avg. units sold (all SKUs, all stores)')
axes[0].set_xticklabels(dow_order, rotation=30, ha='right')

# Month
month_avg = df.groupby(['month_num','month_name'])['quantity_sold'].mean().reset_index()
month_avg = month_avg.sort_values('month_num')
bars2 = axes[1].bar(month_avg['month_name'], month_avg['quantity_sold'],
                    color=[PALETTE[0] if m in [11,12,1,2,3] else PALETTE[2]
                           for m in month_avg['month_num']],
                    edgecolor='white', linewidth=0.5)
axes[1].bar_label(bars2, fmt='%.1f', padding=3, fontsize=9)
axes[1].set_title('Average Daily Sales by Month')
axes[1].set_ylabel('Avg. units sold')

# Legend for season
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=PALETTE[0], label='Rainy season (Nov–Mar)'),
              Patch(facecolor=PALETTE[2], label='Dry season (Apr–Oct)')]
axes[1].legend(handles=legend_els, fontsize=9)

plt.tight_layout()
plt.savefig('../data/eda_temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

sat_lift = (dow_avg['Saturday'] / dow_avg[['Monday','Tuesday','Wednesday','Thursday']].mean() - 1) * 100
print(f'\nSaturday sales lift vs. Mon–Thu average: +{sat_lift:.1f}%')

---
## 4. Bolivian Seasonality — Rainy vs. Dry Season

In [ ]:
df['season'] = df['month_num'].apply(lambda m: 'Rainy (Nov–Mar)' if m in [11,12,1,2,3] else 'Dry (Apr–Oct)')

# Category × Season interaction
season_cat = df.groupby(['category','season'])['quantity_sold'].mean().unstack()
season_cat['lift_rainy_pct'] = (season_cat['Rainy (Nov–Mar)'] / season_cat['Dry (Apr–Oct)'] - 1) * 100
season_cat = season_cat.sort_values('lift_rainy_pct', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Seasonal Demand — Rainy vs. Dry Season by Category', fontsize=13, fontweight='bold')

# Grouped bar
x = np.arange(len(season_cat))
w = 0.35
axes[0].bar(x - w/2, season_cat['Rainy (Nov–Mar)'], w, label='Rainy season', color=PALETTE[0], alpha=0.85)
axes[0].bar(x + w/2, season_cat['Dry (Apr–Oct)'], w, label='Dry season', color=PALETTE[2], alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(season_cat.index, rotation=35, ha='right')
axes[0].set_ylabel('Avg. daily units sold')
axes[0].set_title('Average Sales by Season')
axes[0].legend()

# Lift chart
colors_lift = [PALETTE[0] if v > 0 else PALETTE[1] for v in season_cat['lift_rainy_pct']]
bars = axes[1].barh(season_cat.index, season_cat['lift_rainy_pct'], color=colors_lift, alpha=0.85)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
axes[1].set_xlabel('Rainy season lift vs. dry season (%)')
axes[1].set_title('Seasonal Lift by Category')

plt.tight_layout()
plt.savefig('../data/eda_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSeasonal lift summary:')
print(season_cat[['Rainy (Nov–Mar)','Dry (Apr–Oct)','lift_rainy_pct']].round(2).to_string())

---
## 5. Holiday Effect Analysis

In [ ]:
HOLIDAYS = {
    '2023-08-06': 'Independencia',
    '2023-06-21': 'Año Nuevo Aymara',
    '2024-08-06': 'Independencia',
    '2024-06-21': 'Año Nuevo Aymara',
    '2025-08-06': 'Independencia',
    '2025-06-21': 'Año Nuevo Aymara',
    '2023-03-23': 'Día del Mar',
    '2024-03-23': 'Día del Mar',
    '2025-03-23': 'Día del Mar',
    '2023-05-01': 'Día del Trabajo',
    '2024-05-01': 'Día del Trabajo',
    '2025-05-01': 'Día del Trabajo',
}

holiday_dates = pd.to_datetime(list(HOLIDAYS.keys()))
global_daily_mean = daily['quantity_sold'].mean()

holiday_effects = []
for date_str, name in HOLIDAYS.items():
    d = pd.to_datetime(date_str)
    window = daily[(daily['date'] >= d - pd.Timedelta('7d')) & (daily['date'] <= d + pd.Timedelta('7d'))].copy()
    window['days_from_holiday'] = (window['date'] - d).dt.days
    window['holiday'] = name
    holiday_effects.append(window)

he_df = pd.concat(holiday_effects)
he_agg = he_df.groupby(['holiday','days_from_holiday'])['quantity_sold'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
key_holidays = ['Independencia', 'Año Nuevo Aymara', 'Día del Mar', 'Día del Trabajo']
for i, holiday in enumerate(key_holidays):
    subset = he_agg[he_agg['holiday'] == holiday]
    ax.plot(subset['days_from_holiday'], subset['quantity_sold'],
            marker='o', lw=2, label=holiday, color=PALETTE[i], markersize=5)

ax.axvline(0, color='black', lw=1.2, linestyle='--', alpha=0.6)
ax.axhline(global_daily_mean, color='gray', lw=1, linestyle=':', alpha=0.7, label='Overall mean')
ax.set_xlabel('Days relative to holiday (0 = holiday)')
ax.set_ylabel('Avg. total units sold')
ax.set_title('Holiday Effect: Sales Window ±7 Days Around Key Bolivian Dates', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks(range(-7, 8))
plt.tight_layout()
plt.savefig('../data/eda_holiday_effect.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Category-Level Deep Dive

In [ ]:
monthly_cat = df.groupby(['year','month_num','category'])['quantity_sold'].sum().reset_index()
monthly_cat['period'] = pd.to_datetime(monthly_cat.apply(lambda r: f"{int(r.year)}-{int(r.month_num):02d}-01", axis=1))

top_cats = df.groupby('category')['quantity_sold'].sum().nlargest(6).index

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Monthly Sales Trend by Category (2023–2025)', fontsize=13, fontweight='bold')

for ax, (cat, color) in zip(axes.flat, zip(top_cats, PALETTE)):
    subset = monthly_cat[monthly_cat['category'] == cat].sort_values('period')
    ax.fill_between(subset['period'], subset['quantity_sold'], alpha=0.2, color=color)
    ax.plot(subset['period'], subset['quantity_sold'], lw=1.8, color=color)
    ax.set_title(cat, fontweight='bold')
    ax.set_ylabel('Units/month')
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('../data/eda_category_trends.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Store Performance Comparison

In [ ]:
STORE_LABELS = {'ST-001': 'La Paz', 'ST-002': 'Santa Cruz', 'ST-003': 'Cochabamba', 'ST-004': 'El Alto'}
df['store_label'] = df['store_id'].map(STORE_LABELS)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Store Performance Analysis', fontsize=14, fontweight='bold')

# Total sales share
store_total = df.groupby('store_label')['quantity_sold'].sum()
axes[0].pie(store_total, labels=store_total.index, autopct='%1.1f%%',
            colors=PALETTE[:4], startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('Sales Share by Store')

# Monthly evolution per store
store_monthly = df.groupby(['store_label','year','month_num'])['quantity_sold'].sum().reset_index()
store_monthly['period'] = pd.to_datetime(store_monthly.apply(
    lambda r: f"{int(r.year)}-{int(r.month_num):02d}-01", axis=1))

for store, color in zip(STORE_LABELS.values(), PALETTE):
    sub = store_monthly[store_monthly['store_label'] == store].sort_values('period')
    axes[1].plot(sub['period'], sub['quantity_sold'], lw=2, label=store, color=color)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[1].set_title('Monthly Sales by Store')
axes[1].set_ylabel('Units/month')
axes[1].legend(fontsize=9)

# Category mix heatmap
pivot = df.groupby(['store_label','category'])['quantity_sold'].sum().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
sns.heatmap(pivot_pct, ax=axes[2], annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, cbar_kws={'label': '% of store sales'})
axes[2].set_title('Category Mix by Store (%)')
axes[2].set_xlabel('')
axes[2].set_ylabel('')
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('../data/eda_store_performance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Year-over-Year Growth & Trend

In [ ]:
yoy = df.groupby(['year','month_num'])['quantity_sold'].sum().reset_index()
yoy_pivot = yoy.pivot(index='month_num', columns='year', values='quantity_sold')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Year-over-Year Analysis', fontsize=14, fontweight='bold')

# Overlay years
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for year, color in zip([2023, 2024, 2025], PALETTE[:3]):
    if year in yoy_pivot.columns:
        axes[0].plot(month_labels, yoy_pivot[year], marker='o', lw=2.2,
                     label=str(year), color=color, markersize=6)
axes[0].set_title('Monthly Sales by Year')
axes[0].set_ylabel('Total units sold')
axes[0].legend()
axes[0].set_xticklabels(month_labels, rotation=30)

# Annual growth
annual = df.groupby('year')['quantity_sold'].sum()
growth = annual.pct_change() * 100
bars = axes[1].bar(annual.index.astype(str), annual.values,
                   color=PALETTE[:3], edgecolor='white', linewidth=0.5, alpha=0.9)
axes[1].bar_label(bars, fmt='%.0f', padding=4, fontsize=10)
for i, (yr, g) in enumerate(growth.items()):
    if not np.isnan(g):
        axes[1].text(i, annual[yr] * 0.5, f'+{g:.1f}% YoY',
                     ha='center', va='center', fontsize=11, color='white', fontweight='bold')
axes[1].set_title('Annual Sales Volume & YoY Growth')
axes[1].set_ylabel('Total units sold')

plt.tight_layout()
plt.savefig('../data/eda_yoy_growth.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Key Findings & Modeling Implications

### 📌 Summary of Findings

| Finding | Implication for Model |
|---|---|
| **Saturday shows +40% lift** over Mon–Thu average | `day_of_week` is a critical feature; use sin/cos encoding |
| **Rainy season (Nov–Mar)** drives Paint & Plumbing up ~2× | `is_rainy_season` binary flag + month cyclical features |
| **Dry season (Apr–Oct)** drives Construction & Tools up ~1.7× | Inverse seasonality features per category |
| **Independencia (Aug 6)** = highest single-day spike (+60%) | Explicit holiday indicator features |
| **~5% zero-sale days** (stockouts / closed) | Handle zeros carefully; separate stockout from true zero-demand |
| **~6.5% compound annual growth** | Include trend feature (year fractional or linear time index) |
| **Santa Cruz & La Paz** account for ~65% of volume | Store-level models or store size multiplier feature |

### 🔧 Feature Engineering Decisions

- **Cyclical encoding** for day, week, month (sin/cos) preserves temporal continuity
- **Lag features** at t−1, t−7, t−14, t−28 capture short and medium-term autocorrelation
- **Rolling means** at 7/14/30-day windows smooth noise and reveal trends
- **Explicit holiday flags** for all major Bolivian dates significantly improve accuracy
- **Store × Category interaction** captured via grouped lag features

### 🤖 Model Selection

XGBoost was selected over alternatives (SARIMA, Prophet, LSTM) for this use case because:
- It handles the tabular feature space naturally (no manual seasonality decomposition needed)
- Robust to the ~5% zero-sale noise without special treatment
- Inference is fast enough for daily batch scoring across 120 SKU×Store combinations
- Interpretable via feature importance — critical for stakeholder trust

---

> **Next step:** Run `src/models/forecasting.py` to train the model and generate the 30-day demand forecast.  
> Full case study: [bravoaidatastudio.com/portfolio](https://bravoaidatastudio.com/portfolio/)
